# Milestone 15 - MCP Testing Tools Server

Milestone 15 adds a standalone FastMCP server that exposes reusable testing tools for the project.

## What MCP Means Here

MCP is the standardized tool layer, not the LangGraph State. LangGraph State remains the workflow memory. MCP provides a consistent way for clients or future agents to call testing tools.

## Why After Local Tools

The local repository, API, and report tools worked first. This milestone exposes selected safe actions through a server so they can be reused through a standard protocol.

## MCP Server Architecture

`Host -> MCP client -> FastMCP server -> testing tools`

Transport: stdio for local development.

## Tools Exposed

- `health_check`
- `validate_url_tool`
- `list_project_files_tool`
- `read_text_file_tool`
- `clone_repository_tool`
- `send_http_request_tool`
- `generate_html_report_tool`
- `save_json_artifact_tool`

## Security Note

No secrets, no arbitrary command execution, no target repository code execution, and mutating HTTP methods are skipped by default.

## Part A - Direct Python Self-Test

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root for MCP server imports:", project_root)


Project root for MCP server imports: C:\Users\malak\Desktop\TEST_AUTO\sma_test_automation


In [2]:
from mcp_servers.testing_tools_server import health_check

health_check()

{'status': 'ok', 'server': 'testing-tools-server', 'tools_version': '0.1.0'}

## Part B - Fake Repository File Tools

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

from mcp_servers.testing_tools_server import list_project_files_tool, read_text_file_tool

with TemporaryDirectory() as tmp:
    repo = Path(tmp) / "fake_repo"
    (repo / "todo").mkdir(parents=True)
    (repo / "templates").mkdir()
    (repo / "README.md").write_text("# Fake API", encoding="utf-8")
    (repo / "todo" / "urls.py").write_text("urlpatterns = []", encoding="utf-8")
    (repo / "templates" / "login.html").write_text("<form></form>", encoding="utf-8")
    files = list_project_files_tool(str(repo))
    readme = read_text_file_tool(str(repo), "README.md")

files, readme

({'status': 'success',
  'repo_path': 'C:\\Users\\malak\\AppData\\Local\\Temp\\tmp5mzjy7ti\\fake_repo',
  'files': ['README.md', 'templates/login.html', 'todo/urls.py'],
  'count': 3,
  'error': None},
 {'status': 'success',
  'repo_path': 'C:\\Users\\malak\\AppData\\Local\\Temp\\tmp5mzjy7ti\\fake_repo',
  'relative_path': 'README.md',
  'content': '# Fake API',
  'chars': 10,
  'error': None})

## Part C - HTTP Tool Safe Mode

In [4]:
from mcp_servers.testing_tools_server import send_http_request_tool

send_http_request_tool("POST", "http://localhost:8000/api/todos/", allow_mutating=False)

{'status': 'skipped',
 'method': 'POST',
 'url': 'http://localhost:8000/api/todos/',
 'status_code': None,
 'duration_ms': None,
 'text_preview': '',
 'json_preview': None,
 'request_headers': {},
 'error': None,
 'error_type': 'safety',
 'details': 'Mutating HTTP methods are disabled by default.'}

## Part D - MCP Client Discovery

In a notebook environment, use `await list_testing_mcp_tool_names()` if the event loop supports stdio subprocess tools. If not, run the self-test command below.

In [5]:
from test_auto.mcp.testing_mcp_client import list_testing_mcp_tool_names

# await list_testing_mcp_tool_names()
print("Fallback command: python mcp_servers/testing_tools_server.py --self-test")

Fallback command: python mcp_servers/testing_tools_server.py --self-test


## Part E - Inspector Instruction

To inspect manually:

`npx @modelcontextprotocol/inspector python mcp_servers/testing_tools_server.py`